# core

> Dockerfile generation, image building, container running, and testing

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re, json, os, yaml, keyring, time
from fastcore.all import listify, joins, is_listy, L, patch, concat, bind, Path, filter_values, run, merge, true

# DockerFile

In [ ]:
#| export
def mk_flags(*a, short=True, sym='=', **kw):
    'Build CLI flag list: single-char kwargs → -k [v], multi-char → --key[=v]. short=False: all keys use --key[=v]'
    k2f = lambda k: f'--{k.rstrip("_").replace("_","-")}'
    flags, sc, uv, vt = list(a), lambda k: len(k) == 1, lambda v: v not in (True, False, None), lambda v: v is True
    if short:
        flags += concat([f'-{k}', str(v)] for k,v in kw.items() if sc(k) and uv(v))
        flags += [f'-{k}' for k,v in kw.items() if sc(k) and vt(v)]
        kw = {k: v for k,v in kw.items() if not sc(k)}
    flags += [f'{k2f(k)}{sym}{v}' for k,v in kw.items() if uv(v)]
    flags += [k2f(k) for k,v in kw.items() if vt(v)]
    return flags

## Dockerfile Instructions

`instr` creates a Dockerfile instruction string from a keyword and value. Factory functions (`from_`, `run_`, `cmd_`, etc.) wrap `instr` for each Dockerfile keyword, handling formatting details like tag joining, JSON exec form, and multi-command chaining.

### Instruction factory functions

Each function maps to a Dockerfile keyword with a trailing `_` to avoid clashing with Python builtins.

In [ ]:
#| export
def _instr(kw, v): return '%s %s'%(kw,v)
def _run(cmd: list | str): return _instr('RUN', joins(' && ', listify(cmd)))
def _cmd(cmd: list | str): return _instr('CMD', json.dumps(cmd) if is_listy(cmd) else cmd)
def _add(src:str, dst:str): return _instr('ADD', f'%s %s'%(src,dst))
def _workdir(path): return _instr('WORKDIR', path)
def _env(k, v=None): return _instr('ENV','%s%s'%(k,f'={v}' if v else ''))
def _arg(nm, def_=None): return _instr('ARG','%s%s'%(nm,f'={def_}' if def_ else ''))
def _expose(port): return _instr('EXPOSE', str(port))
def _entrypoint(cmd): return _instr('ENTRYPOINT', json.dumps(cmd) if is_listy(cmd) else cmd)
def _label(**kw): return _instr('LABEL', joins(' ', (f'{k}="{v}"' for k, v in kw.items())))
def _user(u): return _instr('USER', u)
def _volume(path): return _instr('VOLUME', json.dumps(path) if is_listy(path) else path)
def _shell(cmd): return _instr('SHELL', json.dumps(cmd))
def _stop_sig_(sig): return _instr('STOPSIGNAL', sig)
def _on_build(ins): return _instr('ONBUILD', str(ins))

def _from(image:str, tag:str=None, as_:str=None) -> str:
	'From instruction -- base image with optional tag and alias'
	t,a = ':%s'%tag if tag else '', ' AS %s'%as_ if as_ else ''
	return _instr('FROM', '%s%s%s'%(image,t,a))

def _copy(src:str, dst:str, from_:str=None, link=False):
	'COPY instruction -- copy files into image'
	return _instr('COPY', ' '.join([*mk_flags(short=False, link=link, from_=from_), src, dst]))

def _apt_install(*pkgs, y=False, clean=True):
	'RUN apt-get update && apt-get install packages'
	f,ps = '-y ' if y else '', " ".join(pkgs)
	c = ' && rm -rf /var/lib/apt/lists/*' if clean else ''
	return _run(['apt-get update', f'apt-get install {f}{ps}{c}'])

def _healthcheck(cmd, i=None, t=None, r=None, sp=None):
	'HEALTHCHECK instruction -- container health check'
	opts = joins(' ', mk_flags(short=False, interval=i, timeout=t, retries=r, start_period=sp))
	c = json.dumps(cmd) if is_listy(cmd) else cmd
	return _instr('HEALTHCHECK', f'{opts} CMD {c}'.strip())

In [ ]:
assert _instr('RUN', 'echo hello') == 'RUN echo hello'
assert _from('python', '3.11') == 'FROM python:3.11'
assert _from('ubuntu', as_='builder') == 'FROM ubuntu AS builder'
assert _from('alpine') == 'FROM alpine'
assert _run('apt-get update') == 'RUN apt-get update'
r = _run(['apt-get update', 'apt-get install -y curl'])
assert 'apt-get update && ' in r
assert 'apt-get install -y curl' in r
assert _apt_install('curl', 'wget', y=True) == 'RUN apt-get update && apt-get install -y curl wget && rm -rf /var/lib/apt/lists/*'
assert _apt_install('git', clean=False) == 'RUN apt-get update && apt-get install git'
assert _cmd(['python', 'app.py']) == 'CMD ["python", "app.py"]'
assert _cmd('echo hello') == 'CMD echo hello'
assert _copy('.', '/app') == 'COPY . /app'
assert _copy('/build/out', '/app', from_='builder') == 'COPY --from=builder /build/out /app'
assert _copy('app/', '.', link=True) == 'COPY --link app/ .'
assert _copy('/app', '/app', from_='builder', link=True) == 'COPY --from=builder --link /app /app'
assert _workdir('/app') == 'WORKDIR /app'
assert _env('PATH', '/usr/local/bin') == 'ENV PATH=/usr/local/bin'
assert _env('DEBIAN_FRONTEND=noninteractive') == 'ENV DEBIAN_FRONTEND=noninteractive'
assert _expose(8080) == 'EXPOSE 8080'
assert _entrypoint(['python', '-m', 'flask']) == 'ENTRYPOINT ["python", "-m", "flask"]'
assert _arg('VERSION', '1.0') == 'ARG VERSION=1.0'
assert _arg('VERSION') == 'ARG VERSION'
assert _label(version='1.0', maintainer='me') == 'LABEL version="1.0" maintainer="me"'
assert _volume('/data') == 'VOLUME /data'
assert _volume(['/data', '/logs']) == 'VOLUME ["/data", "/logs"]'
assert _shell(['/bin/bash', '-c']) == 'SHELL ["/bin/bash", "-c"]'
assert 'CMD curl' in _healthcheck('curl -f http://localhost/', i='30s')
assert _healthcheck('curl localhost', i='30s', t='10s') == 'HEALTHCHECK --interval=30s --timeout=10s CMD curl localhost'
assert _healthcheck('curl localhost') == 'HEALTHCHECK CMD curl localhost'
assert _on_build(_run('echo triggered')) == 'ONBUILD RUN echo triggered'

## Dockerfile Builder

The `Dockerfile` class provides a fluent interface for building Dockerfiles. Start with a base image, chain instruction methods, then render or save.

Each method is one line -- it creates an instruction and appends it, returning `self` for chaining.

In [ ]:
#| export
def _parse(path_or_str: Path | str):
    'Parse Dockerfile text into list of instruction strings'
    t = Path(path_or_str).read_text() if isinstance(path_or_str, Path) else path_or_str
    return L.splitlines(re.sub(r'\\\n\s*', '', t)).filter(lambda l: l.strip() and not l.strip().startswith('#'))

In [ ]:
parsed = _parse("# comment\nFROM python:3.11\nRUN apt-get update && \\\n    apt-get install -y curl\nCOPY . /app")
print(parsed)
assert len(parsed) == 3
assert parsed[0] == 'FROM python:3.11'
assert 'apt-get install -y curl' in parsed[1]

['FROM python:3.11', 'RUN apt-get update && apt-get install -y curl', 'COPY . /app']


In [ ]:
#| export
class Dockerfile(L):
    'Fluent builder for Dockerfiles'
    @classmethod
    def load(cls, path:Path=Path('Dockerfile')): return cls(_parse(Path(path)))
    def from_(self, base, tag=None, as_=None): return self._add(_from(base, tag, as_))
    def _add(self, i): return self._new(self.items + [i])
    def run(self, cmd): return self._add(_run(cmd))
    def cmd(self, cmd): return self._add(_cmd(cmd))
    def copy(self, src, dst, from_=None, link=False): return self._add(_copy(src, dst, from_, link))
    def add(self, src, dst): return self._add(_add(src, dst))
    def workdir(self, path='/app'): return self._add(_workdir(path))
    def env(self, key, value=None): return self._add(_env(key, value))
    def expose(self, port): return self._add(_expose(port))
    def entrypoint(self, cmd): return self._add(_entrypoint(cmd))
    def arg(self, name, default=None): return self._add(_arg(name, default))
    def label(self, **kwargs): return self._add(_label(**kwargs))
    def user(self, user): return self._add(_user(user))
    def volume(self, path): return self._add(_volume(path))
    def shell(self, cmd): return self._add(_shell(cmd))
    def healthcheck(self, cmd, **kw): return self._add(_healthcheck(cmd, **kw))
    def stopsignal(self, signal): return self._add(_stop_sig_(signal))
    def onbuild(self, instruction): return self._add(_on_build(instruction))
    def apt_install(self, *pkgs, y=True, clean=True):
	    return self._add(_apt_install(*pkgs, y=y, clean=clean))

    def run_mount(self, cmd, type='cache', target=None, **kw):
        'RUN --mount=... for build cache mounts (uv, pip, apt) and secrets'
        opts = f'type={type}'
        if target: opts += f',target={target}'
        for k, v in kw.items(): opts += f',{k.replace("_", "-")}={v}'
        return self._add(f'RUN --mount={opts} {cmd}')

    def __call__(self, inst, *args, **kw):
        'Build a generic Dockerfile instruction: kw ARG1 ARG2 --flag=val --bool-flag'
        rest = joins(' ', [*mk_flags(short=False, **kw), *map(str, args)])
        return self._add(f'%s %s'%(inst.upper(), rest))

    def __getattr__(self, nm):
        'Dispatch unknown instruction names: df.some_instr(arg) → SOME-INSTR arg'
        if nm.startswith('_'): raise AttributeError(nm)
        return bind(self, nm.upper().rstrip('_'))

    def __str__(self): return joins(chr(10),self)
    def __repr__(self): return str(self)

    def save(self, path:Path=Path('Dockerfile')):
        Path(path).mk_write(str(self))
        return self

In [ ]:
df = (Dockerfile().from_('python:3.11-slim')
    .run('pip install flask')
    .copy('.', '/app')
    .workdir('/app')
    .expose(5000)
    .cmd(['python', 'app.py']))

expected = """FROM python:3.11-slim
RUN pip install flask
COPY . /app
WORKDIR /app
EXPOSE 5000
CMD [\"python\", \"app.py\"]"""

print(df)
assert str(df) == expected

FROM python:3.11-slim
RUN pip install flask
COPY . /app
WORKDIR /app
EXPOSE 5000
CMD ["python", "app.py"]


In [ ]:
# run_mount: cache mounts for fast rebuilds
df = (Dockerfile().from_('python:3.12-slim')
    .run_mount('pip install -r requirements.txt', target='/root/.cache/pip')
    .run_mount('uv sync --frozen', target='/root/.cache/uv')
    .run_mount('apt-get install -y curl', type='cache', target='/var/cache/apt'))
s = str(df)
assert "RUN --mount=type=cache,target=/root/.cache/pip pip install -r requirements.txt" in s
assert "RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen" in s
print(df)

FROM python:3.12-slim
RUN --mount=type=cache,target=/root/.cache/pip pip install -r requirements.txt
RUN --mount=type=cache,target=/root/.cache/uv uv sync --frozen
RUN --mount=type=cache,target=/var/cache/apt apt-get install -y curl


Multi-stage builds work naturally:

In [ ]:
df = (Dockerfile().from_('golang:1.21', as_='builder')
    .workdir('/src')
    .copy('.', '.')
    .run('go build -o /app')
    .from_('alpine')
    .copy('/app', '/app', from_='builder')
    .cmd(['/app']))

assert 'FROM golang:1.21 AS builder' in str(df)
assert 'COPY --from=builder /app /app' in str(df)
print(df)

FROM golang:1.21 AS builder
WORKDIR /src
COPY . .
RUN go build -o /app
FROM alpine
COPY --from=builder /app /app
CMD ["/app"]


Multi-command RUN chains with `&&`:

In [ ]:
df = (Dockerfile().from_('ubuntu:22.04').run(['apt-get update', 'apt-get install -y python3', 'rm -rf /var/lib/apt/lists/*']))
print(df)

FROM ubuntu:22.04
RUN apt-get update && apt-get install -y python3 && rm -rf /var/lib/apt/lists/*


### Loading from an existing Dockerfile

Use `Dockerfile.load()` to read an existing Dockerfile. `save()` returns the `Path` it wrote to.

In [ ]:
import tempfile

In [ ]:
tmp = tempfile.mkdtemp()
Path(f'{tmp}/Dockerfile').write_text("# My app\nFROM python:3.11-slim\nRUN apt-get update && \\\n    apt-get install -y curl\nCOPY . /app\nCMD [\"python\", \"app.py\"]")

# Load existing Dockerfile
df = Dockerfile.load(f'{tmp}/Dockerfile')
assert len(df) == 4
assert df[0] == 'FROM python:3.11-slim'

# save writes the file and returns self (Dockerfile), not the path
p = f'{tmp}/Dockerfile'
df.save(p)
assert Path(p).exists()

# chain after loading
df2 = df.run('echo hi')
assert len(df2) == 5
print(df)

FROM python:3.11-slim
RUN apt-get update && apt-get install -y curl
COPY . /app
CMD ["python", "app.py"]


## Build, Run, Test

These top-level functions wrap the Docker CLI for the common workflow: build an image from a `Dockerfile`, run a container, and test that a command succeeds inside an image.

### Requires Docker daemon

The functions below need a running Docker daemon.

In [ ]:
#| export
class Cli:
    'Base: __call__ builds flags → _run(), __getattr__ dispatches subcommands'
    def __call__(self, cmd, *a, **kw): return self._run(cmd, *mk_flags(*a, **kw))
    def _run(self, cmd, *a): raise NotImplementedError
    def __getattr__(self, nm):
        if nm.startswith('_'): raise AttributeError(nm)
        return bind(self, nm.replace('_', '-'))

In [ ]:
#| export
def _docker_cmd(*args, cli='docker'): return run(cli or 'docker', *args)

def _compose_cmd():
	'Return compose invocation style: ("docker", "compose") or ("docker-compose",)'
	try: return ('docker-compose',) if run('docker-compose', 'version') else ('docker', 'compose')
	except Exception: return 'docker', 'compose'

def _clean_cfg():
	'Create a docker config dir with credential helpers stripped, symlinking contexts and cli-plugins'
	src = Path(os.environ.get('DOCKER_CONFIG', Path.home()/'.docker'))
	dst = Path.home()/'.fastops'/'config'
	cfgf = dst/'config.json'
	if not cfgf.exists():
		dst.mkdir(parents=True, exist_ok=True)
		cfg = src.joinpath('config.json').read_json() if (src/'config.json').exists() else {}
		cfg.pop('credsStore', None); cfg.pop('credHelpers', None)
		cfgf.write_json(cfg)
	for name in ('contexts', 'cli-plugins'):
		link, target = dst/name, src/name
		if target.exists() and not link.exists(): link.symlink_to(target)
	return str(dst)

class Docker(Cli):
	'Unified docker CLI wrapper. no_creds strips credential helpers from config.'
	def __init__(self, no_creds=False): self.no_creds = no_creds
	def _run(self, cmd, *a): return _docker_cmd(*self._pre(), cmd, *a)

	def _pre(self):
		chk = os.environ.get('DOCKR_RUNTIME', 'docker') == 'docker' and self.no_creds
		return ('--config', _clean_cfg()) if chk else ()

	def compose(self, *a, f='docker-compose.yml'):
		'Run docker compose subcommand — auto-detects docker-compose vs docker compose'
		base, *sub = _compose_cmd()
		pre = self._pre() if sub else ()  # --config is a docker CLI flag; docker-compose doesn't support it
		return _docker_cmd(*pre, *sub, '-f', f, *a, cli=base)


In [ ]:
#| export
@patch
def build(df:Dockerfile, tag:str=None, path:str='.', no_creds=False, fn='Dockerfile'):
    'Build image from Dockerfile via docker compose build (uses daemon BuildKit, no buildx required).'
    import subprocess, tempfile
    df.save(Path(path) / fn)
    svc = {'build': {'context': str(Path(path).resolve())}}
    if tag: svc['image'] = tag
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yml', delete=False) as f:
        f.write(yaml.dump({'services': {'img': svc}})); tmp = f.name
    try:
        pre = ['--config', _clean_cfg()] if no_creds else []
        res = subprocess.run(['docker'] + pre + ['compose', '-f', tmp, 'build'], capture_output=True)
        if res.returncode: raise IOError((res.stdout + b' ;; ' + res.stderr).decode().strip())
    finally:
        Path(tmp).unlink(missing_ok=True)
    return tag

def test(img_or_tag:str, cmd):
	'Run cmd in image, return True if exit code 0'
	try: Docker().run('--rm', str(img_or_tag), *listify(cmd)); return True
	except Exception: return False

def _container_running(name:str, timeout:float=2.0, poll:float=0.1) -> bool:
	'Poll until container is running or has exited. Returns True if running.'
	deadline = time.monotonic() + timeout
	while time.monotonic() < deadline:
		try: status = Docker()('inspect', '--format={{.State.Status}}', name).strip()
		except IOError: return False  # already removed (--rm) or never created
		if status == 'running': return True
		if status in ('exited', 'dead', 'removing'): return False
		time.sleep(poll)
	return True  # timed out without definitive state — optimistic

def drun(img:str, detach=False, ports=None, name=None, remove=False, command=None, check=True):
	'Run a container, return container ID (detached) or output. check=True raises on startup failure.'
	p_flags = concat(['-p', f'{hp}:{cp.split("/")[0]}'] for cp, hp in (ports or {}).items())
	result = Docker()('run', *p_flags, *mk_flags(d=detach, rm=remove, name=name), str(img), *listify(command or []))
	if detach and check:
		nm = name or result.strip()
		if not _container_running(nm):
			try: log = Docker()('logs', nm)
			except Exception: log = '(no logs available)'
			raise RuntimeError(f"Container {nm!r} failed to start.\n{log}")
	return result

In [ ]:
from fastcore.test import test_fail

In [ ]:
# Debug: see what exception drun actually raises
try:
    drun('alpine', detach=True, command=['sh', '-c', 'exit 1'], check=True)
    print('NO EXCEPTION RAISED')
except Exception as e:
    print(f'Exception type: {type(e).__name__}')
    print(f'Exception message: {str(e)[:300]}')

Exception type: RuntimeError
Exception message: Container '578084b28be52a324fd50dc429702be1a47132fc89cfe73c9525251755bbfe30' failed to start.



### Convenience functions

In [ ]:
#| export
def containers(all=False): return Docker().ps(format='{{.Names}}', a=all).splitlines()
def images(): return Docker().images(format='{{.Repository}}:{{.Tag}}').splitlines()
def stop(nm): Docker()('stop', nm)
def logs(nm, n=10): return Docker()('logs', f'--tail={n}', nm)
def rm(nm, force=False): Docker()('rm', *mk_flags(f=force), nm)
def rmi(nm, force=False): Docker()('rmi', *mk_flags(f=force), nm)

### Example: FastHTML app with uv

A realistic Dockerfile for a [FastHTML](https://fastht.ml) app that uses `uv` for dependency management, installs system packages, and is designed to run with a mounted volume for persistent data.

In [ ]:
df = (Dockerfile().from_('python', '3.12-slim')
    .apt_install('curl', 'sqlite3', y=True)
    .run('pip install uv')
    .workdir('/app')
    .copy('pyproject.toml', '.')
    .run('uv export --no-hashes -o requirements.txt && pip install -r requirements.txt')
    .copy('.', '.')
    .volume('/app/data')
    .expose(5001)
    .cmd(['python', 'main.py']))

print(df)

FROM python:3.12-slim
RUN apt-get update && apt-get install -y curl sqlite3 && rm -rf /var/lib/apt/lists/*
RUN pip install uv
WORKDIR /app
COPY pyproject.toml .
RUN uv export --no-hashes -o requirements.txt && pip install -r requirements.txt
COPY . .
VOLUME /app/data
EXPOSE 5001
CMD ["python", "main.py"]


In [ ]:
#| eval: false
tmp = tempfile.mkdtemp()
df = Dockerfile().from_('alpine').run('echo hello > /greeting.txt').cmd(['cat', '/greeting.txt'])
tag = df.build(tag='fastops-test:hello', path=tmp, no_creds=True)
print(f'Built: {tag}')
out = drun(tag, remove=True)
print(f'Output: {out}')
rmi(tag, force=True)
print('Cleaned up.')

Built: fastops-test:hello
Output: hello
Cleaned up.


### End-to-end: FastHTML + FastLite todo app

In [ ]:
app_dir = Path(tempfile.mkdtemp()) / 'fasthtml-todo'
app_dir.mkdir()

code = {'main.py' :'''
from json import dumps
from fasthtml.common import *

db = database('data/todos.db')
todos = db.t.todos
if todos not in db.t: todos.create(id=int, title=str, done=bool, pk='id')
Todo = todos.dataclass()

app, rt = fast_app(live=False)

@rt('/')
def get():
    items = [Li(f"{'✓' if t.done else '○'} {t.title}", id=f'todo-{t.id}') for t in todos()]
    return Titled('Todos',
        Ul(*items),
        Form(Input(name='title', placeholder='New todo...'), Button('Add'), action='/add', method='post'))

@rt('/add', methods=['post'])
def post(title: str):
    todos.insert(Todo(title=title, done=False))
    return Redirect('/')

@rt('/api/todos')
def api():
    data = [dict(id=t.id, title=t.title, done=t.done) for t in todos()]
    return Response(dumps(data), media_type='application/json')

serve(host='0.0.0.0', port=5001)
''', 'requirements.txt':'python-fasthtml\n'}

for k,v in code.items(): Path(app_dir / k).write_text(v)

# Add some todos via POST
def test_post(url='http://localhost:5001'):
	for t in ['Buy milk', 'Write docs', 'Ship fastops']: urlread(f'{url}/add', title=t)
	# Fetch the JSON API
	for t in urljson(f'{url}/api/todos'): print(f"  {'✓' if t['done'] else '○'} {t['title']}")

def cleanup(nm, tag):
	rm(nm, force=True)
	rmi(tag, force=True)

print(f'App dir: {app_dir}')
print('Files:', os.listdir(app_dir))

App dir: /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmphf0q8sp4/fasthtml-todo
Files: ['requirements.txt', 'main.py']


In [ ]:
df = (Dockerfile()
    .from_('python', '3.12-slim')
    .workdir('/app')
    .copy('requirements.txt', '.')
    .run('pip install --no-cache-dir -r requirements.txt')
    .copy('.', '.')
    .volume('/app/data')
    .expose(5001)
    .cmd(['python', 'main.py']))

print(df)

FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
VOLUME /app/data
EXPOSE 5001
CMD ["python", "main.py"]


In [ ]:
import time
from fastcore.net import urlread, urljson

In [ ]:
#| eval: false
tag = 'fastops-fasthtml:latest'
name = 'fastops-fasthtml-demo'
df.build(tag=tag, path=str(app_dir), no_creds=True)
print(f'Built: {tag}')
for cid in Docker()('ps', '-aq', '--filter', f'ancestor={tag}').splitlines(): rm(cid, force=True)
rm(name, force=True)
cid = drun(tag, detach=True, ports={'5001/tcp': 5001}, name=name, check=True)
print(f'Container: {cid[:12]}')
time.sleep(3)

try: test_post()
except Exception as e:
    print(f'Test failed: {e}')
    print(f'\nLogs:')
    print(logs(name, n=3))
finally: cleanup(name, tag)
print('Cleaned up.')

Built: fastops-fasthtml:latest
Container: bf1125b54374
  ○ Buy milk
  ○ Write docs
  ○ Ship fastops
Cleaned up.


# Docker Compose
> A builder for `docker-compose.yml` files, with a fluent interface for defining services, networks, and volumes. Use `up()` and `down()` to run the stack directly from the builder.

## Service
A `service` is a component of a Docker Compose stack, defined by its image or build context, ports, environment variables, volumes, and other configuration. The `service()` function creates a service definition that can be added to a `Compose` builder.

In [ ]:
#| export
def dict2str(d: dict, sep=':'): return [f'{k}{sep}{v}' for k, v in d.items()] if isinstance(d, dict) else d
def service(image=None, build=None, ports=None, env=None, volumes=None, depends_on=None, command=None, **kw):
	'Create a docker-compose service dict'
	if isinstance(build, Dockerfile): build = '.'
	return filter_values(dict(image=image, command=command, depends_on=depends_on,
	                          ports=dict2str(ports), build=build, environment=dict2str(env, '='),
	                          volumes=dict2str(volumes)), lambda v: v is not None) | kw

In [ ]:
d = service(image='nginx', ports={80: 80})
assert d['image'] == 'nginx'
assert d['ports'] == ['80:80']

In [ ]:
d = service(image='postgres:15', env={'POSTGRES_PASSWORD': 'secret'}, volumes={'pgdata': '/var/lib/postgresql/data'})
assert d['environment'] == ['POSTGRES_PASSWORD=secret']
assert d['volumes'] == ['pgdata:/var/lib/postgresql/data']

## Compose

The `Compose` class provides a fluent builder for docker-compose files. Chain `.svc()`, `.network()`, and `.volume()` calls, then render with `str()` or save to disk.

Services are stored as plain dicts. `to_dict()` just assembles the top-level compose structure.

In [ ]:
#| export
class Compose(L):
	'Fluent builder for docker-compose.yml files'
	def _add(self, item): return self._new(self.items + [item])
	def _collect(self, tag): return {n: c for t, n, c in self if t == tag}
	def save(self, path='docker-compose.yml'): Path(path).write_text(str(self)); return self
	def svc(self, name, **kw): return self._add(('services', name, service(**kw)))
	def network(self, name, **kw): return self._add(('networks', name, kw or None))
	def volume(self, name, **kw): return self._add(('volumes', name, kw or None))

	@classmethod
	def load(cls, path='docker-compose.yml'):
		'Load an existing docker-compose.yml'
		d = yaml.safe_load(Path(path).read_text())
		it = [('services', n, c) for n, c in (d.get('services') or {}).items()]
		it += [('networks', n, c) for n, c in (d.get('networks') or {}).items()]
		it += [('volumes', n, c) for n, c in (d.get('volumes') or {}).items()]
		return cls(it)

	def to_dict(self):
		'Render full compose config as dict'
		d = {'services': self._collect('services')}
		if nets := self._collect('networks'): d['networks'] = nets
		if vols := self._collect('volumes'):  d['volumes'] = vols
		return d

	def up(self, detach=True, path='docker-compose.yml', no_creds=False, **kw):
		'Run docker compose up'
		self.save(path)
		return Docker(no_creds).compose('up', *mk_flags(d=detach, **kw), f=path)

	def down(self, path='docker-compose.yml', no_creds=False, **kw):
		'Run docker compose down'
		return Docker(no_creds).compose('down', *mk_flags(**kw), f=path)

	def logs(self, path='docker-compose.yml', no_creds=False, **kw):
		'Run docker compose logs'
		return Docker(no_creds).compose('logs', *mk_flags(**kw), f=path)

	def __str__(self): return yaml.dump(self.to_dict(), default_flow_style=False, sort_keys=False)
	def __repr__(self): return str(self)

In [ ]:
dc = (Compose()
      .svc('web', image='nginx', ports={80: 80}, networks=['backend'])
      .svc('redis', image='redis:alpine')
      .svc('db', image='postgres:15', env={'POSTGRES_PASSWORD': 'secret'},
           volumes={'pgdata': '/var/lib/postgresql/data'})
      .network('backend').volume('pgdata'))

d = dc.to_dict()
assert 'networks' in d
assert 'volumes' in d
assert 'pgdata' in d['volumes']
print(dc)

services:
  web:
    image: nginx
    ports:
    - 80:80
    networks:
    - backend
  redis:
    image: redis:alpine
  db:
    image: postgres:15
    environment:
    - POSTGRES_PASSWORD=secret
    volumes:
    - pgdata:/var/lib/postgresql/data
networks:
  backend: null
volumes:
  pgdata: null



### Compose up, test, down, and load from an existing docker-compose.yml


In [ ]:
dc = (Compose()
      .svc('app',
           build='.',
           ports={5001: 5001},
           restart='unless-stopped',
           volumes={'app_data': '/app/data'})
      .volume('app_data')).save(app_dir/'docker-compose.yml')
assert (app_dir/'docker-compose.yml').exists()
print((app_dir/'docker-compose.yml').read_text())

services:
  app:
    ports:
    - 5001:5001
    build: .
    volumes:
    - app_data:/app/data
    restart: unless-stopped
volumes:
  app_data: null



In [ ]:
from fastcore.foundation import working_directory

In [ ]:
#| eval: false
with working_directory(app_dir) as w:
    dc.up(no_creds=True)
    time.sleep(3)
    test_post()
    dc.down(v=True,rmi='all',remove_orphans=True)
    print('\n Compose load test: \n')
    print(Compose.load())

  ○ Buy milk
  ○ Write docs
  ○ Ship fastops

 Compose load test: 

services:
  app:
    ports:
    - 5001:5001
    build: .
    volumes:
    - app_data:/app/data
    restart: unless-stopped
volumes:
  app_data: null



## App Builders

Two uv strategies available as `@patch` methods on `Dockerfile`:

- `inst_uv()` — single-stage: copies uv binary from its official image, syncs deps in place
- `with_uv(uv_image, image, workdir)` — multistage: full uv compile stage → slim runtime, only `.venv` copied across

`packages(*pkgs)` wraps `apt_install` but is a no-op when called with no args, so callers can always write `.packages(*listify(pkgs))` without an `if pkgs` guard.

In [ ]:
#| export
@patch
def inst_uv(self:Dockerfile, req=False, wd='/app'):
	'Single-stage uv install: copy uv binary and sync deps. req=True uses requirements.txt instead of pyproject.toml'
	self = self.copy('/uv', '/usr/local/bin/uv', from_='ghcr.io/astral-sh/uv:latest')
	if req: return self.copy('requirements.txt', '.').run('uv pip install --system -r requirements.txt')
	return self.copy('pyproject.toml', '.').run('uv sync --no-dev').env('PATH', f'{wd}/.venv/bin:$PATH')

@patch
def with_uv(self:Dockerfile, uv_image, image, workdir):
	'Multistage uv builder: appends uv builder stage then runtime base onto df'
	return (self.from_(uv_image, as_='builder').workdir(workdir)
			.env('UV_COMPILE_BYTECODE', '1').env('UV_LINK_MODE', 'copy')
			.copy('pyproject.toml', '.').copy('uv.lock', '.')
			.run_mount('uv sync --frozen --no-dev --no-install-project', target='/root/.cache/uv')
			.copy('.', '.').run_mount('uv sync --frozen --no-dev', target='/root/.cache/uv')
			.from_(image).workdir(workdir))

@patch
def packages(self:Dockerfile, *pkgs):
	'Install system packages with apt-get. pkgs can be a list or space-separated string.'
	return self.apt_install(*pkgs) if pkgs else self

### Framework builders

| Function | Default port | Strategy |
|---|---|---|
| `python_app` | 8000 | multistage uv builder → slim |
| `fasthtml_app` | 5001 | `python_app` bound to port 5001 |
| `fastapi_react` | 8000 | Node frontend + Python backend |
| `go_app` | 8080 | Go builder → distroless |
| `rust_app` | 8080 | Cargo builder → distroless |
| `node_app` | 3000 | Node slim (or two-stage static serve) |
| `detect_app` | auto | Sniffs project files, delegates above |

`fasthtml_app` is `python_app` partially applied to `port=5001` via `bind`.

In [ ]:
#| export
def python_app(port=8000, cmd=None, im='python:3.13-slim', wd='/app', pkgs=None,
    vols=None, multistage=True, uv_image='ghcr.io/astral-sh/uv:python3.13-bookworm', healthcheck=None, req=False):
    'Python app Dockerfile. multistage=True (default): uv-builder → slim. False: single-stage with uv binary copy. req=True: use requirements.txt instead of pyproject.toml (forces multistage=False).'
    if multistage:
        df = Dockerfile().with_uv(uv_image, im, wd).copy(wd, wd, from_='builder').env('PATH', f'{wd}/.venv/bin:$PATH').packages(*listify(pkgs))
    else: df = Dockerfile().from_(im).workdir(wd).packages(*listify(pkgs)).inst_uv(req=req, wd=wd).copy('.', '.')
    if vols: df = df.run('mkdir -p ' + ' '.join(listify(vols)))
    if healthcheck: df = df.healthcheck(f'curl -f http://localhost:{port}{healthcheck}', i='30s', t='5s', r='3')
    _cmd = cmd or ['python', 'main.py']
    return df.expose(port).cmd(_cmd if isinstance(_cmd, list) else _cmd.split())

def fastapi_react(port=8000, node_version='20', frontend_dir='frontend', build_dir='dist',
    image='python:3.13-slim', pkgs=None, healthcheck='/health', multistage=False,
    uv_image='ghcr.io/astral-sh/uv:python3.13-bookworm'):
    'Two-stage (default) or three-stage (multistage=True) Dockerfile: Node.js frontend + Python/FastAPI backend'
    df = (Dockerfile().from_(f'node:{node_version}-slim', as_='frontend')
          .workdir('/build').copy(f'{frontend_dir}/package*.json', '.')
          .run('npm ci').copy(frontend_dir, '.').run('npm run build'))
    if multistage: df = df.with_uv(uv_image, image, '/app').copy('/app', '/app', from_='builder').env('PATH', '/app/.venv/bin:$PATH').packages(*listify(pkgs))
    else: df = df.from_(image).workdir('/app').packages(*listify(pkgs)).inst_uv(wd='/app').copy('.', '.')
    df = df.copy(f'/build/{build_dir}', '/app/static', from_='frontend')
    if healthcheck: df = df.healthcheck(f'curl -f http://localhost:{port}{healthcheck}', i='30s', t='5s', r='3')
    return df.expose(port).cmd(['uvicorn', 'main:app', '--host', '0.0.0.0', f'--port={port}'])

def go_app(port=8080, go_version='1.22', binary='app', runtime='gcr.io/distroless/static', cmd=None, cgo=False):
    'Two-stage Go Dockerfile: go compiler + go mod cache → distroless runtime'
    df = (Dockerfile().from_(f'golang:{go_version}-alpine', as_='builder')
          .workdir('/src').copy('go.mod', '.').copy('go.sum', '.')
          .run_mount('go mod download', target='/go/pkg/mod')
          .copy('.', '.').env('CGO_ENABLED', '0' if not cgo else '1')
          .run(f'go build -ldflags="-s -w" -o /{binary} .').from_(runtime)
          .copy(f'/{binary}', f'/{binary}', from_='builder').expose(port))
    return df.cmd(cmd or [f'/{binary}'])

def rust_app(port=8080, rust_version='1', binary='app', runtime='gcr.io/distroless/static', features=None):
    'Two-stage Rust Dockerfile: cargo build --release → distroless runtime'
    build_cmd = 'cargo build --release' + (f' --features {features}' if features else '')
    df = (Dockerfile().from_(f'rust:{rust_version}-slim-bookworm', as_='builder').workdir('/src')
          .copy('.', '.').run_mount(build_cmd, target='/usr/local/cargo/registry').from_(runtime)
          .copy(f'/src/target/release/{binary}', f'/{binary}', from_='builder').expose(port))
    return df.cmd([f'/{binary}'])

def node_app(port=3000, node_version='20', cmd=None, build_cmd='npm run build', static=False):
    'Node.js Dockerfile. static=True does two-stage build → serve dist/; False runs node directly.'
    df = (Dockerfile().from_(f'node:{node_version}-slim', as_='builder' if static else None)
          .workdir('/app').copy('package*.json', '.').run('npm ci').copy('.', '.'))
    if static:
        return (df.run(build_cmd).from_(f'node:{node_version}-slim').workdir('/app')
                .run('npm install -g serve').copy('/app/dist', '.', from_='builder')
                .expose(port).cmd(cmd or ['serve', '-s', '.', '-l', str(port)]))
    return df.expose(port).cmd(cmd or ['node', 'index.js'])

fasthtml_app = bind(python_app, port=5001, cmd=['python', 'app.py'], multistage=False)

def detect_app(path='.', multistage=True, use_railpack=False, **kw):
    'Detect project type and return the appropriate Dockerfile. use_railpack=True falls back to rp_build() for unsupported stacks (PHP, Java, Ruby, .NET, Elixir, Deno, C++ etc.) — returns tag str instead of Dockerfile in that case. For **kw see each app builder.'
    p = Path(path)
    has = lambda f: (p/f).exists()
    read = lambda f: (p/f).read_text().lower() if has(f) else ''
    pyp, req= has('pyproject.toml'), has('requirements.txt')
    has_py = pyp or req
    if has('Cargo.toml'): return rust_app(**kw)
    if has('go.mod'): return go_app(**kw)
    if has('package.json') and has_py: return fastapi_react(multistage=multistage, **kw)
    if has('package.json'): return node_app(**kw)
    if pyp and 'python-fasthtml' in read('pyproject.toml'): return fasthtml_app(multistage=multistage, **kw)
    if pyp: return python_app(multistage=multistage, **kw)
    if req: return python_app(multistage=False, req=True, **kw)
    if use_railpack: return rp_build(path=path, **kw)
    raise ValueError(f'Cannot detect project type in {path!r}. Try use_railpack=True for PHP, Java, Ruby, .NET, Elixir, Deno etc.')

## Railpack integration

[Railpack](https://railpack.com) builds images via BuildKit LLB — no Dockerfile needed. It auto-detects 12+ languages including PHP, Java, Ruby, .NET, Elixir, Deno and more. Requires the `railpack` binary and a running BuildKit daemon.

In [ ]:
#| export
def rp_plan(path='.'):
    'Run `railpack plan` on path and return the parsed build plan dict. Useful for inspecting detected language, packages, and build steps without building.'
    result = run('railpack', 'plan', str(Path(path).resolve()))
    return json.loads(result)

def rp_build(path='.', tag=None, buildkit_host=None, config=None):
    'Build image from path using railpack + BuildKit. Supports all railpack-detected languages (PHP, Java, Ruby, .NET, Elixir, Deno, C++ etc.). config dict is written as railpack.json if one does not already exist. Returns tag.'
    import subprocess
    p = Path(path).resolve()
    cfg_path = p / 'railpack.json'
    wrote_cfg = False
    if config and not cfg_path.exists():
        cfg_path.write_text(json.dumps(config, indent=2))
        wrote_cfg = True
    try:
        args = ['railpack', 'build']
        if tag: args += ['--tag', tag]
        args.append(str(p))
        env = os.environ.copy()
        if buildkit_host: env['BUILDKIT_HOST'] = buildkit_host
        res = subprocess.run(args, env=env, capture_output=True)
        if res.returncode: raise IOError((res.stdout + b' ;; ' + res.stderr).decode().strip())
    finally:
        if wrote_cfg: cfg_path.unlink(missing_ok=True)
    return tag

### App builder tests

In [ ]:
df = Dockerfile().from_('python:3.13-slim').workdir('/app').inst_uv()
s = str(df)
assert 'COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv' in s
assert 'uv sync --no-dev' in s
assert 'ENV PATH=/app/.venv/bin:$PATH' in s
assert 'COPY uv.lock' not in s
print('inst_uv OK')

df = Dockerfile().with_uv('ghcr.io/astral-sh/uv:python3.13-bookworm', 'python:3.13-slim', '/app')
s = str(df)
assert 'uv:python3.13-bookworm AS builder' in s
assert 'ENV UV_COMPILE_BYTECODE=1' in s and 'ENV UV_LINK_MODE=copy' in s
assert 'FROM python:3.13-slim' in s
print('build_uv OK')

assert len(Dockerfile().from_('alpine').packages()) == 1   # no-op
assert 'apt-get install -y curl' in str(Dockerfile().from_('alpine').packages('curl'))
print('packages OK')

df = Dockerfile().from_('python:3.13-slim').workdir('/app').inst_uv(req=True)
s = str(df)
assert 'COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv' in s
assert 'COPY requirements.txt .' in s
assert 'uv pip install --system -r requirements.txt' in s
assert 'pyproject.toml' not in s and 'uv sync' not in s
assert 'ENV PATH' not in s
print('inst_uv(req=True) OK')

In [ ]:
df = python_app(); s = str(df)
assert 'uv:python3.13-bookworm AS builder' in s and 'ENV UV_COMPILE_BYTECODE=1' in s
assert 'COPY --from=builder /app /app' in s and 'ENV PATH=/app/.venv/bin:$PATH' in s
assert 'FROM python:3.13-slim' in s and 'EXPOSE 8000' in s and 'CMD ["python", "main.py"]' in s
print('python_app() multistage OK')

df = python_app(port=5001, multistage=False, pkgs=['libpq-dev', 'curl'], vols=['/app/data'], healthcheck='/health')
s = str(df)
assert 'COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv' in s
assert 'apt-get install -y libpq-dev curl' in s
assert 'ENV PATH=/app/.venv/bin:$PATH' in s
assert 'mkdir -p /app/data' in s and 'HEALTHCHECK' in s and 'EXPOSE 5001' in s
print('python_app() single-stage OK')

df = python_app(multistage=False, req=True); s = str(df)
assert 'COPY requirements.txt .' in s
assert 'uv pip install --system -r requirements.txt' in s
assert 'pyproject.toml' not in s
assert 'ENV PATH' not in s
print('python_app(req=True) OK')

In [ ]:
df = fasthtml_app(); s = str(df)
assert 'EXPOSE 5001' in s and 'uv sync --no-dev' in s and 'CMD ["python", "app.py"]' in s
assert 'ENV PATH=/app/.venv/bin:$PATH' in s and 'AS builder' not in s
df2 = fasthtml_app(pkgs=['rclone'], vols=['/app/data'], healthcheck='/health'); s2 = str(df2)
assert 'rclone' in s2 and 'mkdir -p /app/data' in s2 and 'HEALTHCHECK' in s2
print('fasthtml_app() OK')

df = fastapi_react(); s = str(df)
assert s.count('FROM') == 2 and 'FROM node:20-slim AS frontend' in s
assert 'COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv' in s
assert 'COPY --from=frontend /build/dist /app/static' in s and 'uvicorn' in s
print('fastapi_react() 2-stage OK')

df = fastapi_react(multistage=True); s = str(df)
assert s.count('FROM') == 3 and 'uv:python3.13-bookworm AS builder' in s
assert 'COPY --from=builder /app /app' in s and 'ENV PATH=/app/.venv/bin:$PATH' in s
assert 'COPY --from=frontend /build/dist /app/static' in s
print('fastapi_react() 3-stage OK')

In [ ]:
df = go_app(); s = str(df)
assert 'FROM golang:1.22-alpine AS builder' in s and 'go mod download' in s
assert 'ENV CGO_ENABLED=0' in s and '-o /app' in s
assert 'FROM gcr.io/distroless/static' in s and 'CMD ["/app"]' in s
df2 = go_app(binary='srv'); s2 = str(df2)
assert '-o /srv' in s2 and 'COPY --from=builder /srv /srv' in s2 and 'CMD ["/srv"]' in s2
print('go_app() OK')

df = rust_app(); s = str(df)
assert 'FROM rust:1-slim-bookworm AS builder' in s and 'cargo build --release' in s
assert 'COPY --from=builder /src/target/release/app /app' in s and 'CMD ["/app"]' in s
df2 = rust_app(binary='myapp', features='postgres'); s2 = str(df2)
assert '--features postgres' in s2 and '/src/target/release/myapp' in s2
print('rust_app() OK')

df = node_app(); s = str(df)
assert 'FROM node:20-slim' in s and 'npm ci' in s and 'EXPOSE 3000' in s and 'CMD ["node", "index.js"]' in s
df2 = node_app(static=True); s2 = str(df2)
assert 'FROM node:20-slim AS builder' in s2 and 'npm run build' in s2
assert 'COPY --from=builder /app/dist .' in s2 and 'serve' in s2
print('node_app() OK')

go_app() OK
rust_app() OK
node_app() OK


In [ ]:
def _tmp(files):
    d = Path(tempfile.mkdtemp())
    for f, c in files.items(): (d/f).write_text(c)
    return d

assert 'golang' in str(detect_app(_tmp({'go.mod': 'module x'})))
assert 'rust' in str(detect_app(_tmp({'Cargo.toml': '[package]'}))).lower()
assert 'EXPOSE 3000' in str(detect_app(_tmp({'package.json': '{}'})))
assert 'uvicorn' in str(detect_app(_tmp({'package.json': '{}', 'pyproject.toml': '[project]'})))
assert 'EXPOSE 5001' in str(detect_app(_tmp({'pyproject.toml': '[project]\ndependencies=["python-fasthtml"]'})))
assert 'EXPOSE 8000' in str(detect_app(_tmp({'pyproject.toml': '[project]'})))
s = str(detect_app(_tmp({'requirements.txt': 'flask\n'})))
assert 'uv pip install --system -r requirements.txt' in s and 'AS builder' not in s
try: detect_app(_tmp({'README.md': ''}))
except ValueError: pass
else: raise AssertionError('Expected ValueError')
print('detect_app() OK')

# Config and Secrets

## Non-secret config

`env_set` / `env_get` store plain config (VPS IP, SSH key path, server name) in `~/.config/fastops/.env` with mode `0600`. Writes are reflected into `os.environ` immediately so the current process sees changes without a restart.

The file uses the same security model as `~/.aws/credentials`, `~/.config/hcloud/cli.toml`, and SSH keys — owner-read-only on disk.

In [ ]:
#| export
_FASTOPS_ENV = Path.home() / '.config' / 'fastops' / '.env'

def env_set(key, value, path=None):
    'Upsert key=value into fastops .env file and os.environ. Returns True if changed.'
    from dotenv import get_key, set_key
    p = Path(path or _FASTOPS_ENV)
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists(): p.touch(); os.chmod(p, 0o600)
    if get_key(str(p), key) != str(value): set_key(str(p), key, str(value))
    os.environ[key] = str(value)

def env_get(key, path=None, default=None):
    'Read key: os.environ first, then fastops .env file, then default.'
    val = os.environ.get(key)
    if val is not None: return val
    from dotenv import get_key
    return get_key(str(Path(path or _FASTOPS_ENV)), key) or default

In [ ]:
tmp = Path(tempfile.mkdtemp()) / '.env'
env_set('_FO_TEST_IP', '1.2.3.4', path=tmp)
assert os.environ['_FO_TEST_IP'] == '1.2.3.4'
assert env_get('_FO_TEST_IP', path=tmp) == '1.2.3.4'
env_set('_FO_TEST_IP', '1.2.3.4', path=tmp)
del os.environ['_FO_TEST_IP']
assert env_get('_FO_TEST_IP', path=tmp) == '1.2.3.4'
assert env_get('_FO_TEST_MISSING', path=tmp, default='fallback') == 'fallback'
env_set('_FO_TEST_IP', '5.6.7.8', path=tmp)
assert env_get('_FO_TEST_IP', path=tmp) == '5.6.7.8'
print('env_set/env_get OK')

Key _FO_TEST_IP not found in /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp2xqc3991/.env.
Key _FO_TEST_MISSING not found in /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp2xqc3991/.env.


env_set/env_get OK


## Secrets

`secret_set` / `secret_get` store sensitive values (API tokens, tunnel tokens).

| Environment | Backend |
|---|---|
| macOS | OS Keychain via `keyring` (AES-256, Secure Enclave on Apple Silicon) |
| Headless Linux / CI / VPS | `os.environ` fallback — inject secrets at boot via cloud-init or systemd `EnvironmentFile=` |

`keyring` is an optional dependency. If unavailable (or no keychain daemon running), `secret_set` silently skips keychain storage and the env var is the source of truth for the current session. Between sessions on a server, secrets must come from the environment.

In [ ]:
#| export
def secret_set(key, value, service='fastops', save=True, path=None):
	'Store secret in OS keychain (if available) and os.environ. Silent fallback on headless/VPS.'
	try: keyring.set_password(service, key, value)
	except Exception: pass
	if save: env_set(key, value, path=path)
	return True

def secret_get(key, service='fastops', default=None, path=None):
	'Read secret: OS keychain → os.environ → default.'
	try: return keyring.get_password(service, key) or default
	except Exception: pass
	return env_get(key, path=path, default=default)

In [ ]:
secret_set('_FO_TEST_TOKEN', 'tok123')
assert os.environ.get('_FO_TEST_TOKEN') == 'tok123'
assert secret_get('_FO_TEST_TOKEN') == 'tok123'
del os.environ['_FO_TEST_TOKEN']
assert secret_get('_FO_TEST_MISSING_XYZ', default='fallback') == 'fallback'
print('secret_set/secret_get OK')

secret_set/secret_get OK


In [ ]:
#| export
def secrets(*keys, service='fastops', path=None):
    'Read multiple secrets from keychain/env into a dict. Warns and skips any that are missing.'
    return filter_values(merge(*L(keys).map(lambda k: {k:secret_get(k, service=service, path=path)})),true)

In [ ]:
secret_set('_FO_TEST_A', 'val_a')
secret_set('_FO_TEST_B', 'val_b')

result = secrets('_FO_TEST_A', '_FO_TEST_B', '_FO_TEST_MISSING')
assert result == {'_FO_TEST_A': 'val_a', '_FO_TEST_B': 'val_b'}
print('secrets() OK')

secrets() OK


# Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()